# Landslide Hazard Score — multi-city (`LANDSLIDES_SITE`)

Combines five normalized terrain and climate layers into a single **landslide hazard
susceptibility score** (0–1) on a canonical **90 m grid**.

> **Why 90 m?** Landslide initiation zones in the configured city are typically 30–200 m wide.
> At 250 m a steep hillslope is averaged out and may fall below the 15° activation threshold.
> 90 m matches the native MERIT HAND resolution and preserves meaningful slope variation
> from the 30 m Copernicus DEM.

## Methodology summary

| Component | Layer | Source | Role |
|-----------|-------|--------|------|
| Slope risk | Copernicus DEM GLO-30 slope (30 m → 250 m) | GEE `COPERNICUS/DEM/GLO30` | Primary driver + gate (45%) |
| Precip trigger | CHIRPS R90p climatology (~5 km → 250 m) | CHIRPS via GEE | Detonante de lluvia (20%) |
| Soil cohesion | SoilGrids clay % 0–30 cm (250 m) | GEE OpenLandMap | Amplificador de presión de poros (15%) |
| Vegetation protection | MODIS NDVI P10 DJF 2015–2024 (250 m) | GEE MODIS MOD13Q1 | Protección de raíces (10%) |
| Drainage accumulation | MERIT Hydro HAND (90 m → 250 m) | GEE MERIT/Hydro | Saturación de ladera (10%) |
| Land cover modifier | Dynamic World mode 2023 (10 m → 250 m) | GEE DW V1 | ±modificador |

### Formula

```
slope_risk  = clamp01((slope_deg - 15) / 20)       # 0 below 15°; 1 at 35°+
precip_risk = minmax_norm(R90p_clim)               # 0–1 across POA
soil_risk   = clamp01(clay_pct / 40)               # 40% clay = max pore pressure
veg_protect = minmax_norm(ndvi_p10)                # higher NDVI = more protection
hand_factor = clamp01(1 - hand_m / 50)             # near drainage = high, ridgetop = 0

H = clamp01(
    0.45 × slope_risk
  + 0.20 × precip_risk  × slope_risk
  + 0.15 × (1 - soil_risk)  × slope_risk   # low cohesion amplifies slope risk
  + 0.10 × (1 - veg_protect) × slope_risk  # bare slope is more susceptible
  + 0.10 × hand_factor × slope_risk        # drainage accumulation amplifies slope risk
)

# Land cover modifiers applied after combination:
# +0.10 if DW class ∈ {6=built, 7=bare} AND slope ≥ 15°
# ×0.85 if DW class ∈ {1=trees, 2=grass, 3=flooded_veg, 5=shrub} AND NDVI P10 > 0.4
```

All terrain-interaction terms are multiplied by `slope_risk` so that flat areas (slope < 15°)
score zero regardless of other inputs — preserving the geotechnical activation threshold.

## 0. Setup

In [ ]:
import math
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rasterio.features
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import folium
import branca.colormap as cm
from pathlib import Path
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from rasterio.warp import reproject as warp_reproject, calculate_default_transform

print('Libraries loaded OK')

## 1. Paths

In [ ]:
# Site configuration — transformation/landslide_hazard city configs + model defaults
import os
import sys
from pathlib import Path

_HERE = Path.cwd().resolve()
_LANDSLIDE_HAZARD = None
for _candidate in [_HERE, *_HERE.parents]:
    _probe = _candidate / "landslide_hazard" if _candidate.name != "landslide_hazard" else _candidate
    if (_probe / "site_config.py").is_file() and (_probe / "config" / "sites").is_dir():
        _LANDSLIDE_HAZARD = _probe
        break
if _LANDSLIDE_HAZARD is None:
    raise FileNotFoundError("Could not locate transformation/landslide_hazard from notebook cwd")

sys.path.insert(0, str(_LANDSLIDE_HAZARD))
from site_config import load_site_config

LANDSLIDE_HAZARD_ROOT = _LANDSLIDE_HAZARD

SITE_SLUG = os.environ.get("LANDSLIDES_SITE", "porto_alegre")
SITE_CONFIG = load_site_config(SITE_SLUG, LANDSLIDE_HAZARD_ROOT)
SITE_ROOT = SITE_CONFIG["paths_abs"]["site_root"]
INPUT_DIR = SITE_CONFIG["paths_abs"]["data_input"]
INTERMEDIATE_DIR = SITE_CONFIG["paths_abs"]["data_intermediate"]
OUTPUT_DIR = SITE_CONFIG["paths_abs"]["data_output"]
OUT_ROOT = SITE_CONFIG["paths_abs"]["out"]
CACHE_DIR = SITE_CONFIG["paths_abs"]["cache"]
STYLES_DIR = SITE_CONFIG["paths_abs"]["styles"]
OUTPUT_PREFIX = SITE_CONFIG["output_prefix"]
SEASON = SITE_CONFIG["season"]
SEASON_LABEL = SITE_CONFIG["season_label"]
START_YEAR = int(SITE_CONFIG["start_year"])
END_YEAR = int(SITE_CONFIG["end_year"])
DW_YEAR = int(SITE_CONFIG.get("dw_year", 2023))
HAZARD_CFG = SITE_CONFIG["hazard"]
PUBLISH_CFG = SITE_CONFIG.get("publish", {})
BAIRRO_CFG = SITE_CONFIG.get("bairro", {})
MODEL_CONFIG_PATH = SITE_CONFIG["model_config_path"]
LAYER_FILES = SITE_CONFIG["layers"]
OUTPUT_FILES = SITE_CONFIG["outputs"]

for _p in (INPUT_DIR, INTERMEDIATE_DIR, OUTPUT_DIR, OUT_ROOT, CACHE_DIR, STYLES_DIR):
    Path(_p).mkdir(parents=True, exist_ok=True)

print(f"Landslide hazard site: {SITE_CONFIG['display_name']} ({SITE_SLUG})")
print(f"Config: {SITE_CONFIG['config_path']}")
print(f"Model defaults: {MODEL_CONFIG_PATH}")
print(f"Season: {SEASON_LABEL} {START_YEAR}-{END_YEAR}")
print(f"Inputs -> {INPUT_DIR}")


In [ ]:
BASE_DIR = LANDSLIDE_HAZARD_ROOT
INPUT_DIR = SITE_CONFIG["paths_abs"]["data_input"]
OUTPUT_DIR = SITE_CONFIG["paths_abs"]["data_output"]
OUT_ROOT = SITE_CONFIG["paths_abs"]["out"]
STYLES_DIR = SITE_CONFIG["paths_abs"]["styles"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Optional neighbourhood polygons (POA). Skipped when path missing.
BARRIOS_PATH = BAIRRO_CFG.get("gpkg_path_abs")
POA_MUN_CODE = str(BAIRRO_CFG.get("mun_code") or "")
BAIRRO_LAYER = BAIRRO_CFG.get("layer") or "BR_bairros_CD2022"
BAIRRO_ENABLED = bool(BAIRRO_CFG.get("enabled")) and BARRIOS_PATH is not None and Path(BARRIOS_PATH).exists()
if BAIRRO_CFG.get("enabled") and not BAIRRO_ENABLED:
    print("Bairro aggregation configured but gpkg missing — set LANDSLIDE_BAIRRO_GPKG or bairro.gpkg_path")

SLOPE_TIF = INPUT_DIR / LAYER_FILES["slope_deg"]
R90P_TIF = INPUT_DIR / LAYER_FILES["r90p"]
CLAY_TIF = INPUT_DIR / LAYER_FILES["clay_pct"]
NDVI_TIF = INPUT_DIR / LAYER_FILES["ndvi_p10"]
HAND_TIF = INPUT_DIR / LAYER_FILES["hand"]
DW_TIF = INPUT_DIR / LAYER_FILES["dw_mode"]

for p in [SLOPE_TIF, R90P_TIF, CLAY_TIF, NDVI_TIF, HAND_TIF, DW_TIF]:
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}]  {p.name}")

print(f"Site: {SITE_CONFIG['display_name']} ({SITE_SLUG})")
print(f"Weights: {HAZARD_CFG.get('weights')}")


## 2. Inspect input layers

Verify CRS, resolution, and value ranges before combining.

In [ ]:
def info(path, label=None):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        nd  = src.nodata
        valid = arr[np.isfinite(arr)]
        if nd is not None: valid = valid[valid != nd]
        res_m = src.res[0] * 111_320
        print(f"{label or path.name:<38} "
              f"shape={src.shape} "
              f"res={res_m:.0f}m "
              f"range=[{valid.min():.2f}, {valid.max():.2f}]")

print(f"{'Layer':<38} {'Shape':>14} {'Res':>8}  {'Range'}")
print('-' * 78)
info(SLOPE_TIF, 'slope_deg (30m)')
info(R90P_TIF,  'R90p_clim (~5km)')
info(CLAY_TIF,  'clay_pct_250m')
info(NDVI_TIF,  'ndvi_p10_djf (250m)')
info(HAND_TIF,  'hand_m (90m)')
info(DW_TIF,    'dw_mode_2023 (10m)')

## 3. Canonical 90 m reference grid

All layers are reprojected/resampled onto this common grid.
90 m matches the native resolution of MERIT HAND and preserves
meaningful slope variation from the 30 m Copernicus DEM.

> For the downstream risk score, the 90 m hazard will be aggregated to 250 m
> when combining with the shared E/V layers.

In [ ]:
LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = SITE_CONFIG["bbox"]

TARGET_RES_M = int(HAZARD_CFG.get("target_resolution_m", 90))
DEG_PER_METRE = 1 / 111_320
TARGET_RES_DEG = TARGET_RES_M * DEG_PER_METRE

REF_CRS = rasterio.CRS.from_epsg(4326)
ref_width = math.ceil((LON_MAX - LON_MIN) / TARGET_RES_DEG)
ref_height = math.ceil((LAT_MAX - LAT_MIN) / TARGET_RES_DEG)
ref_transform = from_origin(LON_MIN, LAT_MAX, TARGET_RES_DEG, TARGET_RES_DEG)

ref_meta = {
    "driver": "GTiff",
    "dtype": "float32",
    "width": ref_width,
    "height": ref_height,
    "count": 1,
    "crs": REF_CRS,
    "transform": ref_transform,
    "nodata": np.nan,
}

print(f"Canonical grid: {ref_height} rows × {ref_width} cols  (at {TARGET_RES_M} m)")
print(f"Resolution    : {TARGET_RES_M} m ({TARGET_RES_DEG:.6f}°)")
print(f"Total pixels  : {ref_height * ref_width:,}")
print(f"BBox          : lon [{LON_MIN}, {LON_MAX}]  lat [{LAT_MIN}, {LAT_MAX}]")


## 4. Resample all layers to 90 m grid

| Layer | Native res | Resampling | Rationale |
|-------|-----------|------------|-----------|
| slope_deg | 30m | `average` | Mean slope within 90m cell |
| R90p_clim | ~5km | `nearest` | Repeat the 5km cell value — no interpolation. Bilinear would create artificial sub-5km gradients that don't exist in the data. |
| clay_pct | 250m | `bilinear` | Upsampling continuous soil variable |
| ndvi_p10 | 250m | `bilinear` | Upsampling continuous vegetation index |
| hand_m | 90m | `bilinear` | Same resolution — only aligns to exact grid |
| dw_mode | 10m | `nearest` | Categorical — preserves class boundaries at 90m |

In [ ]:
def to_grid(src_path, resampling_method, nodata_val=np.nan):
    """Warp src_path onto the canonical 250m grid, return float32 array."""
    with rasterio.open(src_path) as src:
        out = np.full((ref_height, ref_width), np.nan, dtype=np.float32)
        warp_reproject(
            source=rasterio.band(src, 1),
            destination=out,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_transform,
            dst_crs=REF_CRS,
            resampling=resampling_method,
        )
    return out

print('Resampling layers to 90m grid...')
slope_arr = to_grid(SLOPE_TIF, Resampling.average)   # 30m  → 90m: mean slope per cell
r90p_arr  = to_grid(R90P_TIF,  Resampling.nearest)   # ~5km → 90m: repeat cell value, no interpolation
clay_arr  = to_grid(CLAY_TIF,  Resampling.bilinear)  # 250m → 90m: continuous upsample
ndvi_arr  = to_grid(NDVI_TIF,  Resampling.bilinear)  # 250m → 90m: continuous upsample
hand_arr  = to_grid(HAND_TIF,  Resampling.bilinear)  # 90m  → 90m: aligns to exact grid
dw_arr    = to_grid(DW_TIF,    Resampling.nearest)   # 10m  → 90m: preserve class labels

print(f'  slope  range: [{np.nanmin(slope_arr):.1f}, {np.nanmax(slope_arr):.1f}]°')
print(f'  R90p   range: [{np.nanmin(r90p_arr):.1f}, {np.nanmax(r90p_arr):.1f}] mm')
print(f'  clay   range: [{np.nanmin(clay_arr):.1f}, {np.nanmax(clay_arr):.1f}] %')
print(f'  ndvi   range: [{np.nanmin(ndvi_arr):.3f}, {np.nanmax(ndvi_arr):.3f}]')
print(f'  hand   range: [{np.nanmin(hand_arr):.1f}, {np.nanmax(hand_arr):.1f}] m')
print(f'  DW unique classes: {np.unique(dw_arr[np.isfinite(dw_arr)]).astype(int)}')
print('Done.')

## 5. Compute hazard components

Each component is transformed to a 0–1 contribution before combining.

In [ ]:
SLOPE_GATE = float(HAZARD_CFG.get('slope_gate_deg', 15))
SLOPE_SAT = float(HAZARD_CFG.get('slope_sat_deg', 35))
CLAY_SAT = float(HAZARD_CFG.get('clay_sat_pct', 40))
HAND_SAT = float(HAZARD_CFG.get('hand_sat_m', 50))
NDVI_DENSE = float(HAZARD_CFG.get('ndvi_dense_threshold', 0.4))

def clamp01(arr):
    return np.clip(arr, 0.0, 1.0)

def minmax_norm(arr):
    """Min-max normalization ignoring NaN."""
    lo, hi = np.nanmin(arr), np.nanmax(arr)
    if hi == lo:
        return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)

# ── Fill NaN gaps with neutral defaults before computing components ────────
# MERIT HAND has tile gaps and NoData over water → fill with 25 m (mid-slope,
# neutral: neither near-drainage accumulation zone nor high ridge).
# Other layers may have sparse gaps at edges → fill conservatively.
#
# Default rationale:
#   hand   25 m  → hand_factor = clamp01(1 - 25/50) = 0.50  (neutral)
#   clay   35 %  → soil_risk   = clamp01(35/40)     = 0.875 (slightly cohesive)
#   ndvi    0.3  → veg_protect = normalized ~0.3     (light vegetation)
#   r90p   median → regional average for missing edge pixels

hand_fill  = np.where(np.isnan(hand_arr),  float(HAZARD_CFG.get('fill', {}).get('hand_m', 25.0)), hand_arr)
clay_fill  = np.where(np.isnan(clay_arr),  float(HAZARD_CFG.get('fill', {}).get('clay_pct', 35.0)), clay_arr)
ndvi_fill  = np.where(np.isnan(ndvi_arr),  float(HAZARD_CFG.get('fill', {}).get('ndvi', 0.3)), ndvi_arr)
r90p_fill  = np.where(np.isnan(r90p_arr),  float(np.nanmedian(r90p_arr)), r90p_arr)

# Report gap coverage
for name, raw in [('hand', hand_arr), ('clay', clay_arr),
                  ('ndvi', ndvi_arr), ('r90p', r90p_arr)]:
    pct_nan = np.isnan(raw).mean() * 100
    if pct_nan > 0:
        print(f'  {name}: {pct_nan:.1f}% NaN → filled with default')

# ── Slope risk: geotechnical gate + ramp ──────────────────────────────────
# NaN slope = outside POA / no DEM coverage → kept as NaN (masks final output)
slope_risk = clamp01((slope_arr - 15.0) / 20.0)

# ── Precipitation trigger ─────────────────────────────────────────────────
precip_risk = minmax_norm(r90p_fill)

# ── Soil cohesion ─────────────────────────────────────────────────────────
soil_risk = clamp01(clay_fill / 40.0)

# ── Vegetation protection ─────────────────────────────────────────────────
veg_protect = minmax_norm(ndvi_fill)

# ── Drainage accumulation ─────────────────────────────────────────────────
hand_factor = clamp01(1.0 - (hand_fill / 50.0))

# Print component statistics
for name, arr in [
    ('slope_risk',   slope_risk),
    ('precip_risk',  precip_risk),
    ('soil_risk',    soil_risk),
    ('veg_protect',  veg_protect),
    ('hand_factor',  hand_factor),
]:
    valid = arr[np.isfinite(arr)]
    pct_active = (valid > 0).mean() * 100
    print(f'{name:<16} mean={valid.mean():.3f}  '
          f'p90={np.percentile(valid,90):.3f}  '
          f'max={valid.max():.3f}  '
          f'active={pct_active:.1f}%')

## 6. Combine into hazard score

All terrain-interaction terms are multiplied by `slope_risk` so that
cells with slope < 15° score zero regardless of other inputs.

In [ ]:
# ── Weighted combination (defaults from models/landslide_hazard/config.yaml) ──
_w = HAZARD_CFG.get("weights", {})
_w_slope = float(_w.get("slope_risk", 0.45))
_w_precip = float(_w.get("precip_risk", 0.20))
_w_cohesion = float(_w.get("low_cohesion", 0.15))
_w_veg = float(_w.get("lack_of_veg", 0.10))
_w_hand = float(_w.get("hand_factor", 0.10))

H = (
    _w_slope * slope_risk
  + _w_precip * precip_risk * slope_risk
  + _w_cohesion * (1 - soil_risk) * slope_risk
  + _w_veg * (1 - veg_protect) * slope_risk
  + _w_hand * hand_factor * slope_risk
)

_mod = HAZARD_CFG.get("modifiers", {})
_bare_built = set(int(x) for x in _mod.get("bare_built_classes", [6, 7]))
_dense_veg = set(int(x) for x in _mod.get("dense_veg_classes", [1, 2, 3, 5]))
_boost = float(_mod.get("bare_built_boost", 0.10))
_dampen = float(_mod.get("dense_veg_factor", 0.85))

bare_built_on_slope = np.isin(dw_arr.astype(int), list(_bare_built)) & (slope_arr >= SLOPE_GATE)
H = np.where(bare_built_on_slope, H + _boost, H)

dense_veg = np.isin(dw_arr.astype(int), list(_dense_veg)) & (ndvi_fill > NDVI_DENSE)
H = np.where(dense_veg, H * _dampen, H)

hazard = clamp01(H)

# Mask only where slope has no data — slope is the primary extent driver
hazard = np.where(np.isfinite(slope_arr), hazard, np.nan)

print(f"Hazard score range: {np.nanmin(hazard):.3f} – {np.nanmax(hazard):.3f}")
print(f"% pixels > 0: {np.nanmean(hazard > 0) * 100:.1f}%")
print(f"Weights used: slope={_w_slope} precip={_w_precip} cohesion={_w_cohesion} veg={_w_veg} hand={_w_hand}")


## 7. Visualize input components

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

panels = [
    (slope_risk,   'Slope Risk\n(clamp((slope-15)/20))', 'OrRd'),
    (precip_risk,  'Precip Risk\n(minmax R90p clim.)',   'Blues'),
    (1-soil_risk,  'Low Cohesion\n(1 - clay/40)',        'YlOrBr'),
    (1-veg_protect,'Lack of Vegetation\n(1 - NDVI P10)', 'YlGn_r'),
    (hand_factor,  'Drainage Accumulation\n(1 - HAND/50)','PuBu'),
    (hazard,       'LANDSLIDE HAZARD\nSCORE (0–1)',       'RdYlGn_r'),
]

for ax, (data, title, cmap) in zip(axes, panels):
    im = ax.imshow(data, cmap=cmap, vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('Landslide Hazard Score — Input Components', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 8. Aggregate to bairros

In [ ]:
poa_gdf = None
if not BAIRRO_ENABLED:
    print("Bairro aggregation skipped (not configured or gpkg missing).")
else:
    neigh = gpd.read_file(BARRIOS_PATH, layer=BAIRRO_LAYER) if BAIRRO_LAYER else gpd.read_file(BARRIOS_PATH)
    if POA_MUN_CODE and "CD_MUN" in neigh.columns:
        neigh = neigh[neigh["CD_MUN"] == POA_MUN_CODE].copy()
    if neigh.crs is None:
        neigh = neigh.set_crs("EPSG:4674")
    poa_gdf = neigh.to_crs("EPSG:4326")

    bairro_map = rasterio.features.rasterize(
        [(geom, i + 1) for i, geom in enumerate(poa_gdf.geometry)],
        out_shape=(ref_height, ref_width),
        transform=ref_transform,
        fill=0,
        dtype="int32",
    )

    scores = []
    for i, row in poa_gdf.iterrows():
        idx = poa_gdf.index.get_loc(i) + 1
        mask = bairro_map == idx
        vals = hazard[mask & np.isfinite(hazard)]
        scores.append(vals.mean() if len(vals) > 0 else np.nan)

    poa_gdf["landslide_hazard_score"] = scores
    print(f"Bairros with valid score: {poa_gdf['landslide_hazard_score'].notna().sum()}")
    name_col = "NM_BAIRRO" if "NM_BAIRRO" in poa_gdf.columns else poa_gdf.columns[0]
    print(poa_gdf.nlargest(5, "landslide_hazard_score")[[name_col, "landslide_hazard_score"]].to_string(index=False))


## 9. Interactive map

In [ ]:
import geemap

def _save_tmp(arr, name):
    """Save a float32 array as a temp TIF using NaN as nodata (Float32 native)."""
    path = OUTPUT_DIR / f'_display_{name}.tif'
    profile = {
        'driver': 'GTiff', 'dtype': 'float32',
        'width': ref_width, 'height': ref_height,
        'count': 1, 'crs': REF_CRS,
        'transform': ref_transform,
        'nodata': float('nan'),
        'compress': 'lzw',
    }
    with rasterio.open(path, 'w', **profile) as dst:
        dst.write(arr.astype(np.float32), 1)
    valid = arr[np.isfinite(arr)]
    print(f'  [{name}] saved → {len(valid):,} valid px, '
          f'range [{valid.min():.3f}, {valid.max():.3f}], '
          f'mean={valid.mean():.3f}')
    return str(path)

# Save all layers
layer_files = [
    (_save_tmp(slope_risk,      'slope_risk'),   '1. Slope Risk',                   False),
    (_save_tmp(precip_risk,     'precip_risk'),  '2. Precip Trigger (R90p)',         False),
    (_save_tmp(1 - soil_risk,   'low_cohesion'), '3. Low Cohesion (clay)',           False),
    (_save_tmp(1 - veg_protect, 'lack_veg'),     '4. Lack of Vegetation (NDVI)',     False),
    (_save_tmp(hand_factor,     'hand_factor'),  '5. Drainage Accumulation (HAND)',  False),
    (_save_tmp(hazard,          'hazard'),       '6. Landslide Hazard Score',        True),
]

lat_c = (LAT_MIN + LAT_MAX) / 2
lon_c = (LON_MIN + LON_MAX) / 2
m = geemap.Map(center=[lat_c, lon_c], zoom=11)
m.add_basemap('OpenStreetMap.Mapnik')

# Add layers one by one — no nodata kwarg so geemap detects NaN automatically
for tif_path, layer_name, visible in layer_files:
    m.add_raster(
        tif_path,
        colormap='rdylgn_r',
        vmin=0, vmax=1,
        layer_name=layer_name,
        visible=visible,
    )

m.add_gdf(
    poa_gdf,
    layer_name='Bairros (mean hazard)',
    style={'color': 'white', 'weight': 0.8, 'fillOpacity': 0},
)

m.add_layer_control()
m


## 10. Export

| File | Content |
|------|---------|
| `outputs.landslide_hazard_score` | Pixel-level hazard score (0–1), 90 m, EPSG:4326 |
| `outputs.landslide_hazard_vector` | Optional neighbourhood mean score GeoPackage |


In [ ]:
# ── Save raster ───────────────────────────────────────────────────────────
out_tif = OUTPUT_DIR / OUTPUT_FILES["landslide_hazard_score"]

write_meta = {
    **ref_meta,
    "compress": "lzw",
    "tiled": True,
    "blockxsize": 256,
    "blockysize": 256,
}
with rasterio.open(out_tif, "w", **write_meta) as dst:
    dst.write(hazard.astype(np.float32), 1)
    dst.set_band_description(1, "landslide_hazard_score_0_1")
print(f"Raster saved: {out_tif}")

# ── Save neighbourhood GeoPackage (optional) ───────────────────────────────
out_gpkg = OUTPUT_DIR / OUTPUT_FILES["landslide_hazard_vector"]
if poa_gdf is not None:
    cols = ["CD_BAIRRO", "NM_BAIRRO", "landslide_hazard_score", "geometry"]
    poa_gdf[[c for c in cols if c in poa_gdf.columns]].to_file(out_gpkg, driver="GPKG")
    print(f"Vector saved: {out_gpkg}")
else:
    print("Vector export skipped (no bairro aggregation).")


## 11. COG + Web Tiles

Publishes the hazard score for web maps:
- **COG** (Cloud Optimized GeoTIFF) for direct raster access
- **Visual tiles** (colorized PNG) for display overlay
- **Value-encoded tiles** (RGB-packed float) for client-side hover lookup

**Value encoding formula:** `score = (R + 256×G + 65536×B) / 10000`

In [ ]:
import subprocess, shutil

SCORE_DIR = OUT_ROOT / "landslide_hazard_score"
SCORE_DIR.mkdir(parents=True, exist_ok=True)

in_tif = OUTPUT_DIR / OUTPUT_FILES["landslide_hazard_score"]
cog_tif = SCORE_DIR / "landslide_hazard_score_90m_cog.tif"
colorized_tif = SCORE_DIR / "landslide_hazard_90m_colorized.tif"
value_encoded_tif = SCORE_DIR / "landslide_hazard_90m_value_encoded_rgb.tif"
tiles_dir = SCORE_DIR / "tiles_visual"
value_tiles_dir = SCORE_DIR / "tiles_values"
colors_txt = STYLES_DIR / "landslide_hazard_colors.txt"

print(f"Publish input: {in_tif}")
print(f"Publish dir  : {SCORE_DIR}")
assert in_tif.exists(), f"Missing score raster: {in_tif}"
assert colors_txt.exists(), f"Missing colors: {colors_txt}"


In [ ]:
# 1. COG
subprocess.run([
    'gdal_translate', str(in_tif), str(cog_tif),
    '-of', 'COG', '-ot', 'Float32',
    '-co', 'COMPRESS=DEFLATE',
    '-co', 'RESAMPLING=NEAREST',
    '-co', 'OVERVIEWS=AUTO'
], check=True, capture_output=True)
print('COG created:', cog_tif)

# 2. Colorized raster
subprocess.run([
    'gdaldem', 'color-relief', str(cog_tif), str(colors_txt), str(colorized_tif),
    '-of', 'GTiff', '-alpha'
], check=True, capture_output=True)
print('Colorized raster created:', colorized_tif)

# 3. Visual XYZ tiles  (--xyz = standard web-map convention, Y origin at north)
# NOTE: do NOT use -x / --tmscompatible — that flips the Y axis (TMS convention)
# and produces tiles at y≈844 instead of the correct y≈1203 for Porto Alegre.
if tiles_dir.exists(): shutil.rmtree(tiles_dir)
subprocess.run([
    'gdal2tiles.py', '--tiledriver=PNG', '--webviewer=none',
    f"--zoom={PUBLISH_CFG.get('tile_zoom', '8-15')}", '--resampling=near', '--xyz',
    str(colorized_tif), str(tiles_dir)
], check=True, capture_output=True)
print('Visual tiles created:', tiles_dir)


In [ ]:
# 4. Value-encoded RGB raster
# Encoding: encoded_int = round(clip(score, 0, 1) * 10000)
#   R = encoded_int        & 0xFF
#   G = (encoded_int >> 8) & 0xFF
#   B = (encoded_int >>16) & 0xFF
# Decode in app: score = (R + 256*G + 65536*B) / 10000
base_expr = (
    "numpy.where(numpy.isnan(A), 0, "
    "numpy.rint(numpy.clip(A, 0, 1) * 10000)).astype(numpy.int64)"
)
subprocess.run([
    'gdal_calc.py',
    '-A', str(cog_tif),
    '--outfile', str(value_encoded_tif),
    '--calc', f'bitwise_and({base_expr}, 255)',
    '--calc', f'bitwise_and(right_shift({base_expr}, 8), 255)',
    '--calc', f'bitwise_and(right_shift({base_expr}, 16), 255)',
    '--type', 'Byte',
    '--NoDataValue', '0',
    '--overwrite',
], check=True, capture_output=True)
print('Value-encoded RGB created:', value_encoded_tif)

# 5. Value-encoded XYZ tiles  (--xyz, same as visual tiles)
if value_tiles_dir.exists(): shutil.rmtree(value_tiles_dir)
subprocess.run([
    'gdal2tiles.py', '--tiledriver=PNG', '--webviewer=none',
    f"--zoom={PUBLISH_CFG.get('tile_zoom', '8-15')}", '--resampling=near', '--xyz',
    str(value_encoded_tif), str(value_tiles_dir)
], check=True, capture_output=True)
print('Value-encoded tiles created:', value_tiles_dir)
